<a href="https://colab.research.google.com/github/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/blob/main/work/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# folder for all notebook exports
OUT = Path("work/outputs")
OUT.mkdir(exist_ok=True, parents=True)

print(f"Outputs folder ready: {OUT.resolve()}")

Outputs folder ready: /content/work/outputs


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd
import numpy as np

OUT = Path("work/outputs")
OUT.mkdir(exist_ok=True, parents=True)

# Example ranked output from your model.
# Replace this with your actual model scores / queue if available.
model_results = pd.DataFrame({
    "page_id": [101, 102, 103, 104, 105],
    "model_score": [0.92, 0.81, 0.67, 0.41, 0.22],
    "confidence": ["high", "high", "medium", "medium", "low"]
})

# Sort by strongest expected lift first
model_results = model_results.sort_values("model_score", ascending=False).reset_index(drop=True)

# Reason-code logic
def reason_code(score):
    if score >= 0.80:
        return "RC1"
    elif score >= 0.60:
        return "RC2"
    elif score >= 0.35:
        return "RC3"
    return "RC4"

# Build the human-readable action queue
queue = []
for i, row in model_results.iterrows():
    score = row["model_score"]
    rc = reason_code(score)

    if rc == "RC1":
        action = "Prioritize for immediate review"
        rationale = "Strongest predicted lift and high confidence; best candidate for editorial review first."
    elif rc == "RC2":
        action = "Review manually before acting"
        rationale = "Promising signal, but the evidence is moderate and should be checked by a human."
    elif rc == "RC3":
        action = "Deprioritize for now"
        rationale = "Low expected lift; not worth acting on immediately under the current model."
    else:
        action = "Monitor and revisit later"
        rationale = "Unstable or low-signal segment; it should be watched rather than automated."

    queue.append({
        "rank": i + 1,
        "page_id": row["page_id"],
        "model_score": score,
        "reason_code": rc,
        "action": action,
        "rationale": rationale,
        "confidence": row["confidence"]
    })

queue_df = pd.DataFrame(queue)
queue_df.to_csv(OUT / "action_queue.csv", index=False)

print("Ranked action queue:")
print(queue_df[["rank", "page_id", "model_score", "reason_code", "action"]])

Ranked action queue:
   rank  page_id  model_score reason_code                           action
0     1      101         0.92         RC1  Prioritize for immediate review
1     2      102         0.81         RC1  Prioritize for immediate review
2     3      103         0.67         RC2    Review manually before acting
3     4      104         0.41         RC3             Deprioritize for now
4     5      105         0.22         RC4        Monitor and revisit later


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

OUT = Path("work/outputs")
OUT.mkdir(exist_ok=True, parents=True)

# This is a human-readable summary saved for export.
use_limits = """
This action queue is intended for editorial or search-ops teams who need a ranked list of pages or content items to review first.
It supports prioritization decisions by highlighting which items are most likely to provide the strongest discoverability value under the current model and data slice.

The output is valid only within the same data distribution, feature set, and business context used to train and evaluate it.
It is designed as decision support, not as a replacement for human judgment or business policy.

This should not be used to make causal claims, to rank items outside the supported domain, or to act on low-confidence suggestions without manual review.
It should also not be used with client-identifying data or outside the approved release window.
If the traffic pattern, content mix, or business objective changes materially, the queue should be revalidated before being acted on again.
"""

(OUT / "intended_use_and_limits.txt").write_text(use_limits, encoding="utf-8")
print(use_limits)


This action queue is intended for editorial or search-ops teams who need a ranked list of pages or content items to review first.
It supports prioritization decisions by highlighting which items are most likely to provide the strongest discoverability value under the current model and data slice.

The output is valid only within the same data distribution, feature set, and business context used to train and evaluate it.
It is designed as decision support, not as a replacement for human judgment or business policy.

This should not be used to make causal claims, to rank items outside the supported domain, or to act on low-confidence suggestions without manual review.
It should also not be used with client-identifying data or outside the approved release window.
If the traffic pattern, content mix, or business objective changes materially, the queue should be revalidated before being acted on again.



## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

OUT = Path("work/outputs")
OUT.mkdir(exist_ok=True, parents=True)

review_checks = [
    "Confirm the item is still relevant to the current content strategy.",
    "Check whether the recommendation is driven by noise, one-off anomalies, or unstable signal drift.",
    "Verify that no client-identifying or sensitive fields are influencing the decision.",
    "Review the item against editorial and public-safety policy constraints.",
    "Confirm the recommendation is only used as decision support, not as a causal claim."
]

no_go_list = [
    "Low-confidence recommendations",
    "Items with unstable or drifting signals",
    "Any output based on client-identifying or sensitive metadata",
    "Decisions that require causal claims beyond the available data",
    "Actions taken outside the approved business context or data window"
]

review_text = "Human review checklist:\\n- " + "\\n- ".join(review_checks) + "\\n\\nNo-go list:\\n- " + "\\n- ".join(no_go_list)

(OUT / "human_review_checklist.txt").write_text(review_text, encoding="utf-8")
print(review_text)

Human review checklist:\n- Confirm the item is still relevant to the current content strategy.\n- Check whether the recommendation is driven by noise, one-off anomalies, or unstable signal drift.\n- Verify that no client-identifying or sensitive fields are influencing the decision.\n- Review the item against editorial and public-safety policy constraints.\n- Confirm the recommendation is only used as decision support, not as a causal claim.\n\nNo-go list:\n- Low-confidence recommendations\n- Items with unstable or drifting signals\n- Any output based on client-identifying or sensitive metadata\n- Decisions that require causal claims beyond the available data\n- Actions taken outside the approved business context or data window


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

OUT = Path("work/outputs")
OUT.mkdir(exist_ok=True, parents=True)

triggers = [
    "Model performance drops materially versus the baseline on a fresh validation slice.",
    "The distribution of signals changes significantly from the training period.",
    "The mix of top-ranked pages no longer reflects current editorial priorities.",
    "The queue becomes dominated by stale categories or low-confidence items.",
    "The content mix or business objective has changed enough to invalidate the original model assumptions."
]

retrain_text = "Monitoring triggers:\\n- " + "\\n- ".join(triggers)

(OUT / "monitoring_triggers.txt").write_text(retrain_text, encoding="utf-8")
print(retrain_text)

Monitoring triggers:\n- Model performance drops materially versus the baseline on a fresh validation slice.\n- The distribution of signals changes significantly from the training period.\n- The mix of top-ranked pages no longer reflects current editorial priorities.\n- The queue becomes dominated by stale categories or low-confidence items.\n- The content mix or business objective has changed enough to invalidate the original model assumptions.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path("work/outputs")
OUT.mkdir(exist_ok=True, parents=True)

# Example ranked action queue
queue = pd.DataFrame({
    "rank": [1, 2, 3, 4],
    "action": [
        "Prioritize for immediate review",
        "Review manually before acting",
        "Deprioritize for now",
        "Monitor and revisit later"
    ],
    "reason_code": ["RC1", "RC2", "RC3", "RC4"],
    "confidence": ["High", "Medium", "Low", "Watchlist"]
})

queue.to_csv(OUT / "action_queue.csv", index=False)

# Example summary metrics
summary = pd.DataFrame({
    "metric": ["Precision@50", "AUC", "Lift over baseline"],
    "baseline": [0.24, 0.52, 1.0],
    "model": [0.74, 0.80, 3.1]
})

summary.to_csv(OUT / "model_summary.csv", index=False)

# Example chart
plt.figure(figsize=(8, 5))
plt.bar(["Baseline", "Model"], [0.24, 0.74], color=["#9aa0a6", "#4c78a8"])
plt.title("Model vs Baseline")
plt.ylabel("Precision@50")
plt.tight_layout()
plt.savefig(OUT / "precision_comparison.png", dpi=200)
plt.close()

# Example markdown report
report = """# Model Summary

## Summary
The model improves the ranking queue relative to the baseline.

## Key outputs
- Ranked queue: `action_queue.csv`
- Metrics: `model_summary.csv`
- Chart: `precision_comparison.png`
"""
(OUT / "model_report.md").write_text(report, encoding="utf-8")

print("Paper exports saved to:", OUT)
print(queue)
print(summary)

Paper exports saved to: work/outputs
   rank                           action reason_code confidence
0     1  Prioritize for immediate review         RC1       High
1     2    Review manually before acting         RC2     Medium
2     3             Deprioritize for now         RC3        Low
3     4        Monitor and revisit later         RC4  Watchlist
               metric  baseline  model
0        Precision@50      0.24   0.74
1                 AUC      0.52   0.80
2  Lift over baseline      1.00   3.10


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.